# 环节 03 · 策略梯度与 Actor-Critic（配套 Notebook）

> 配套长文：[环节03-策略梯度与Actor-Critic详解.md](./环节03-策略梯度与Actor-Critic详解.md)
> 定位：手推**似然比技巧**、验证**基线无偏**、量化**方差削减**、跑通 **GAE 反向递推**。全部**纯 Python 标准库**。

**怎么跑**

- 依赖：无。§3 的方差表要 20 万次采样，跑起来约 1~3 秒。

**地图：本 Notebook ↔ 长文章节**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 似然比技巧 | §3 | `∇logπ = onehot − π`（数值验证） |
| §2 REINFORCE | §4 / §10 | 三臂老虎机收敛到最优臂 |
| §3 基线无偏 | §5 | 减基线不改变期望梯度 |
| §4 方差削减 | §5 | 0.3134 → 0.1465（降到 47%） |
| §5 GAE | §7 | λ=0 / 0.95 / 1 三档与反向递推 |
| §6 熵正则 | §8 | 熵的数值与"过早一根筋" |


## 1. 似然比技巧：`∇_θ log π_θ(a|s)` 长什么样

对 softmax 策略 `π = softmax(θ)`，可以证明（也是整个策略梯度的地基）：

```
∇_θ log π_θ(a) = onehot(a) − π
```

- 对被采样到的那个动作：梯度是 `1 − π(a)`（正）；
- 对其他动作：梯度是 `−π(a')`（负）。


In [ ]:
import math

def softmax(th):
    m = max(th)
    e = [math.exp(t - m) for t in th]
    s = sum(e)
    return [x / s for x in e]

theta = [0.5, -0.3, 0.1]
pi = softmax(theta)
a = 2                                             # 采样到动作 2
h = 1e-6
print("π =", [round(p, 4) for p in pi])

# 数值微分：对 θ_i 加一个极小扰动，看 log π(a) 的变化率
num = []
for i in range(3):
    t2 = theta[:]
    t2[i] += h
    num.append((math.log(softmax(t2)[a]) - math.log(pi[a])) / h)

analytic = [(1.0 if i == a else 0.0) - pi[i] for i in range(3)]
print("数值 ∇logπ =", [round(x, 4) for x in num])
print("解析 onehot−π =", [round(x, 4) for x in analytic])
print("→ 只有被采样到的动作有正梯度，其余全被压低。")


## 2. REINFORCE：用回报当权重，放大/缩小动作概率

`θ ← θ + α·A·∇logπ`，其中 `A = R − b`。**R 是权重、不是拟合目标**——这就是它可以任意平移（减基线）的原因。


In [ ]:
import statistics

P = [0.2, 0.5, 0.8]                               # 三个臂的真实成功率（智能体不知道）
ALPHA, EPISODES = 0.1, 3000

def run(use_baseline, seed):
    import random
    random.seed(seed)
    th, b = [0.0, 0.0, 0.0], 0.0
    for _ in range(EPISODES):
        pi = softmax(th)
        r, acc, a = random.random(), 0.0, 0
        for a, p in enumerate(pi):
            acc += p
            if r <= acc:
                break
        R = 1.0 if random.random() < P[a] else 0.0
        adv = R - (b if use_baseline else 0.0)
        grad = [(1.0 if i == a else 0.0) - pi[i] for i in range(3)]
        for i in range(3):
            th[i] += ALPHA * adv * grad[i]
        b += 0.05 * (R - b)
    return softmax(th)

for use_baseline in (False, True):
    runs = [run(use_baseline, seed) for seed in range(5)]
    avg = [statistics.fmean(p[i] for p in runs) for i in range(3)]
    tag = "有基线" if use_baseline else "无基线"
    print(f"  {tag}：收敛 π = [{avg[0]:.3f} {avg[1]:.3f} {avg[2]:.3f}]   （最优臂 p = 0.8）")
print("→ 都收敛到「几乎总是选臂 3」：基线不改最优解，只改到达速度与稳定度。")


## 3. 基线的无偏性：为什么减去一个不依赖动作的量是安全的

```
E_{a~π}[ ∇logπ(a) · b(s) ] = b(s)·Σ_a π(a)·∇logπ(a)
                           = b(s)·Σ_a ∇π(a)          # ∇π = π∇logπ
                           = b(s)·∇(1) = 0            # 概率归一化 ∎
```

所以 `E[g]` 完全不受 `b` 影响。下面用 20 万次采样验证。


In [ ]:
import random

pi = [1/3, 1/3, 1/3]                              # 均匀初始策略
VEC = [[(1.0 if i == a else 0.0) - pi[i] for i in range(3)] for a in range(3)]
N = 200000

def grad_stats(b, seed=1):
    random.seed(seed)
    mean = [0.0, 0.0, 0.0]
    e_norm2 = 0.0
    for _ in range(N):
        k = random.choices(range(3), weights=pi)[0]
        R = 1.0 if random.random() < P[k] else 0.0
        for i in range(3):
            mean[i] += (R - b) * VEC[k][i]
        e_norm2 += sum(((R - b) * VEC[k][i]) ** 2 for i in range(3))
    mean = [x / N for x in mean]
    e_norm2 /= N
    sig = sum(x * x for x in mean)
    return e_norm2, sig, e_norm2 - sig

print("基线 b      E[||g||²]   ||E[g]||²（真实信号）   方差")
for b in (0.0, 0.5, 0.8):
    e2, sig, var = grad_stats(b)
    print(f"  b = {b:<5} {e2:9.4f} {sig:13.4f} {var:14.4f}")
print("→ 三行的真实信号几乎相同（≈0.02）= 无偏性验证；方差从 0.3134 降到 0.1465（47%）。")


## 4. 语义升级：从"绝对回报"到"优势"

```
R（回报）        → "这条轨迹拿了多少分"        （绝对好坏，噪声大）
A = R − b（优势）→ "比平均水平好多少"          （相对好坏，噪声小）
```

**基线只降方差、不改方向** —— 这就是 GRPO 敢用一个"组内均值"当基线的理论许可。


## 5. GAE：把"偏差-方差"做成一个可调旋钮

```
δ_t = r_t + γV(s_{t+1}) − V(s_t)                       ← 单步 TD 误差
A_t = δ_t + γλ·δ_{t+1} + (γλ)²·δ_{t+2} + …             ← 定义
A_t = δ_t + γλ·A_{t+1}   （从后往前递推，O(T)）          ← 实现
```

- λ = 0 → 只看单步（高偏差、低方差）；
- λ = 1 → 等价于 MC 减基线（无偏、高方差）；
- λ = 0.95 → PPO 的实践甜点。


In [ ]:
g = 0.9
rewards = [0.0, 0.0, 0.0, 0.0, 1.0]                # r_0 … r_4
V = [0.5, 0.4, 0.3, 0.2, 0.0]                      # 假想的 critic 估计

deltas = [rewards[t] + g * V[t + 1] - V[t] for t in range(4)]
print("单步 TD 误差 δ =", [round(d, 4) for d in deltas])

for lam in (0.0, 0.95, 1.0):
    A, advs = 0.0, [0.0] * 4
    for t in reversed(range(4)):
        A = deltas[t] + g * lam * A
        advs[t] = A
    print(f"λ = {lam:<5} A = {[round(x, 4) for x in advs]}")
print("→ λ 越大，优势越「往后看」，越接近 MC；GAE 只是把这个折中用指数加权写出来。")


## 6. 熵正则：防止策略过早变成"一根筋"

策略梯度只会强化已知的好动作 → 分布越来越尖（熵越来越小）→ 停止探索 → 卡在平庸解。
加一项 `+ c·H(π)` 就是把"保持多样性"写进目标。熵的最大值是 `ln|A|`。


In [ ]:
def entropy(p):
    return -sum(x * math.log(x) for x in p if x > 0)

print("均匀分布的理论最大熵 ln(3) =", round(math.log(3), 4))
for p in ([0.33, 0.33, 0.34], [0.5, 0.3, 0.2], [0.9, 0.05, 0.05], [1.0, 0.0, 0.0]):
    print(f"  π = {p}   熵 = {entropy(p):.4f}")
print("→ 熵坍缩（越来越小）是策略梯度的头号故障；LLM 长 CoT 训练里由 DAPO 的 Clip-Higher 来治。")


## 7. 小结与下钻

- **似然比技巧**让"不可微的回报"能回传梯度：`∇J = E[∇logπ · R]`。
- **基线无偏、只降方差**：这是优势 `A` 与 GRPO 组基线的合法来源。
- **Actor-Critic** 用网络估 `V` 当基线；**GAE** 用 λ 把偏差-方差调成旋钮。
- **熵正则**防止过早收敛。LLM 里这一切落到"逐 token 策略梯度"。

下一站：[环节 04 · 信赖域与 PPO](./环节04-信赖域与PPO详解.md)（给策略梯度加上"别改太猛"）。
